In [11]:
from rdkit import Chem
from torch_geometric.data import Data
import torch.nn.functional as F
from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)
from torch_geometric.loader import DataLoader


convert molecule to graph

In [12]:
def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum()
        ])

    edge_index = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index.append([i, j])
        edge_index.append([j, i])

    x = torch.tensor(
        x,
        dtype=torch.float
    )

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t()

    return Data(
        x=x,
        edge_index=edge_index
    )

In [13]:
graph = smiles_to_graph(
    "CCO"
)

print(graph)

Data(x=[3, 1], edge_index=[2, 4])


create the GNN

In [14]:
class MoleculeEncoder(
    torch.nn.Module
):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(
            1,
            32
        )

        self.conv2 = GCNConv(
            32,
            64
        )

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = F.relu(x)

        embedding = global_mean_pool(
            x,
            batch
        )

        return embedding

run one molecule

In [15]:
graph = smiles_to_graph(
    "CCO"
)

loader = DataLoader(
    [graph],
    batch_size=1
)

model = MoleculeEncoder()

for batch in loader:

    emb = model(
        batch.x,
        batch.edge_index,
        batch.batch
    )

    print(
        emb.shape
    )

torch.Size([1, 64])
